<a href="https://colab.research.google.com/github/marcelalozano27-ship-it/bsan6200-assignment5/blob/main/Colab%20Notebook/Assignment_5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Marcela Lozano
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid / Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [14]:
# ── Install packages (uncomment as needed) ──
!pip install -q chromadb sentence-transformers huggingface-hub python-dotenv

import os
import re
import pandas as pd
import requests
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
load_dotenv()


print("Imports working")

Imports working


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [15]:
# ── Load JD metadata ──
jd_metadata = pd.read_csv(
    "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/jd_metadata.csv"
)

print(jd_metadata)

                                             filename                Company  \
0                 jd_LAClippers_data_analyst_lead.txt            LA Clippers   
1         jd_alo_production_supplyplanning_intern.txt                    ALO   
2                jd_axs_associate_product_manager.txt                    AXS   
3          jd_ephemeris_financial_strategy_intern.txt             Ephemeris    
4                       jd_fedex_analytics_intern.txt                 Fedex    
5           jd_revolve_data_analyst_merchandising.txt                Revolve   
6       jd_roku_content_analytics_insights_intern.txt                   Roku   
7       jd_skechers_strategic_partnerships_intern.txt               Skechers   
8   jd_sony_insights_research_analytics_summerinte...                   Sony   
9      jd_tiktok_strategic_partner_manager_intern.txt                 TikTok   
10         jd_tinder_corporate_rotational_analyst.txt                 Tinder   
11  jd_universalmusicgroup_Brand_label_o

In [16]:
# ── Load JD documents and resume ──
from langchain_core.documents import Document
import requests

jd_documents = []

base_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/job_descriptions/"

for _, row in jd_metadata.iterrows():
    filename = row["filename"]
    url = base_url + filename

    text = requests.get(url).text

    doc = Document(
        page_content=text,
        metadata={
            "filename": filename,
            "company": row.get("company", ""),
            "title": row.get("title", ""),
            "source_url": row.get("source_url", ""),
            "date_collected": row.get("date_collected", ""),
            "doc_type": "job_description"
        }
    )

    jd_documents.append(doc)

print("Loaded job descriptions:", len(jd_documents))
print(jd_documents[0].metadata)
print(jd_documents[0].page_content[:500])

Loaded job descriptions: 13
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': '', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to optimization recommendations that increase conversion and yield. You will partner closely with Marketing, Sales and Finance stakeholders to translate data into clear actions and measurable outcomes.

T


In [17]:
resume_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/resume/resume.txt"

resume_text = requests.get(resume_url).text

resume_doc = Document(
    page_content=resume_text,
    metadata={
        "filename": "resume.txt",
        "doc_type": "resume"
    }
)

print("Loaded resume")
print(resume_doc.page_content[:500])

Loaded resume
MARCELA LOZANO
Los Angeles, CA
marcelalozano27@gmail.com
linkedin.com/in/marcelalozano

EDUCATION
Loyola Marymount University
M.S. Business Analytics, Expected Aug 2026
GPA: 4.0

University of Texas at Austin
Certification: Data Science and Data Management Systems
Skills: SQL, Python, Tableau, Google Cloud Platform

Loyola Marymount University
B.A. Economics, Minor Spanish
Magna Cum Laude, GPA: 3.89
Valedictorian of Economics Major

TECHNICAL SKILLS
Python
SQL
Tableau
Excel
Pandas
NumPy
scikit-l


In [18]:
# ── Preview sample content ──
all_documents = jd_documents + [resume_doc]

print("Total documents including Resume:", len(all_documents))
for i in range(3):
    print(f"\n--- Job Description {i} ---")
    print(jd_documents[i].metadata)
    print(jd_documents[i].page_content[:300])

Total documents including Resume: 14

--- Job Description 0 ---
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': '', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to o

--- Job Description 1 ---
{'filename': 'jd_alo_production_supplyplanning_intern.txt', 'company': '', 'title': 'Production & Supply Planning Intern', 'source_url': 'https://www.aloyoga.com/pages/careers', 'date_collected': '4/26/2026', 'doc_type': 'job_description'}

We are seeking a motivated and detail-oriented Production & Supply Planning Intern to 

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [19]:
# ── Strategy 1 ── Fixed size chunking

def chunk_text_fixed(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


fixed_chunks = []

for doc in all_documents:
    chunks = chunk_text_fixed(doc.page_content, chunk_size=800, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "fixed_size"
        })

print("Strategy 1 chunks:", len(fixed_chunks))
print(fixed_chunks[0]["text"][:300])

Strategy 1 chunks: 56
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [20]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=800, overlap_words=20):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())

            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return [c for c in chunks if len(c) > 50]


sentence_chunks = []

for doc in all_documents:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=800, overlap_words=20)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "sentence_aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(sentence_chunks[0]["text"][:300])

Strategy 2 chunks: 51
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [21]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    "strategy": ["fixed_size", "sentence_aware"],
    "chunk_size": [800, 800],
    "overlap": ["100 characters", "20 words"],
    "num_chunks": [len(fixed_chunks), len(sentence_chunks)]
})

comparison

,strategy,chunk_size,overlap,num_chunks
0,fixed_size,800,100 characters,56
1,sentence_aware,800,20 words,51


### Chunking Decision

**Which strategy did you choose?**  
Sentence Aware Chunking

**Why?**  
I'm using sentence aware chunking to preserve complete thoughts and sentences within the job descriptions. This helps improve the quality of retrieved information. Fixed size chunking splits important requirements across chunks and is therefore less meaningful for the analysis.  Sentence aware chunking (strategy 2) also provides slightly less chunks than strategy 1 making it a more efficient segmentation of the data.

**Final settings (chunk_size, overlap):**
chunk_size = 800
overlap = 20 words

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


client = chromadb.Client()
collection = client.create_collection(name="job_fit", get_or_create=True)

# use BEST chunks
all_chunks = sentence_chunks

# create embeddings + store
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=False)

# add to chroma
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=all_chunks,
    ids=[str(i) for i in range(len(texts))]
)

print("Vector DB created with", len(texts), "chunks")

In [23]:
# ── Verify: run a test similarity search ──

results = collection.query(
    query_texts=["Python SQL data analyst skills"],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc[:300])


--- Result 1 ---
Strong analytical and problem-solving skills. Proficiency in Microsoft Excel and basic data analysis techniques. Excellent written and verbal communication skills. Ability to work collaboratively in a fast-paced, team-oriented environment. Ability to work full-time onsite in Long Beach, CA. Preferre

--- Result 2 ---
at Austin Certification: Data Science and Data Management Systems Skills: SQL, Python, Tableau, Google Cloud Platform Loyola Marymount University B.A. Economics, Minor Spanish
Magna Cum Laude, GPA: 3.89
Valedictorian of Economics Major

TECHNICAL SKILLS
Python
SQL
Tableau
Excel
Pandas
NumPy
scikit-l

--- Result 3 ---
MARCELA LOZANO
Los Angeles, CA
marcelalozano27@gmail.com
linkedin.com/in/marcelalozano

EDUCATION
Loyola Marymount University
M.S. Business Analytics, Expected Aug 2026
GPA: 4.0

University of Texas at Austin
Certification: Data Science and Data Management Systems
Skills: SQL, Python, Tableau, Googl


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [24]:
# ── Initialize LLM ──
from openai import OpenAI

# load environment variables
load_dotenv(".env")

# initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("LLM initialized")

LLM initialized


In [25]:
# ── Analysis 1: Skill Gap Report ──
def get_skill_gap_analysis(query):
    results = collection.query(
        query_texts=[query],
        n_results=8
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
You are a job fit analyzer. Use ONLY the provided context from the job description and resume.

Task:
Compare the job description against the candidate resume and create a Skill Gap Report.

Rules:
- Do not assume skills that are not stated.
- Do not say "assuming," "not confirmed," or "uncertain."
- Make a clear decision based only on the available text.
- Before listing a skill as missing, first check whether it appears anywhere in the resume context.
- If SQL, Python, Tableau, Excel, Pandas, NumPy, scikit-learn, or analytics experience appear in the resume, list them as matching skills, not gaps.
- Keep missing skills limited to truly absent or weakly demonstrated areas.
- When listing matching skills, cite exact evidence from the resume, such as tools, coursework, projects, or work experience.
- Provide practical recommendations for closing each gap.

Return the response in this structure:

1. Matching skills found in the resume
- Skill:
- Resume evidence:
- Why it matches the JD:

2. Required skills from the job description
- List the major required skills from the JD.

3. Missing or weak skills
- Gap:
- Why it is a gap:

4. Recommended actions to close each gap
- Gap:
- Recommended action:

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You compare resumes to job descriptions using only the provided text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [26]:
analysis = get_skill_gap_analysis(
    "resume skills SQL Python Tableau Excel candidate experience"
)
print(analysis)

1. Matching skills found in the resume
- Skill: SQL
- Resume evidence: "Skills: SQL, Python, Tableau, Google Cloud Platform"
- Why it matches the JD: The job description requires SQL for querying and analyzing datasets, which is explicitly listed in the resume.

- Skill: Python
- Resume evidence: "Skills: SQL, Python, Tableau, Google Cloud Platform"
- Why it matches the JD: Python is mentioned in the resume, aligning with the job's preference for hands-on ML experience.

- Skill: Tableau
- Resume evidence: "Skills: SQL, Python, Tableau, Google Cloud Platform"
- Why it matches the JD: The job description mentions data visualization tools such as Tableau, which is included in the candidate's skill set.

- Skill: Excel
- Resume evidence: "Working knowledge of Microsoft Word, Excel, Outlook and PowerPoint."
- Why it matches the JD: The job description emphasizes strong proficiency in Excel, which the candidate demonstrates.

- Skill: Data visualization
- Resume evidence: "Solid experience 

In [27]:
# ── Analysis 2: Keyword Alignment ──
def get_keyword_alignment(query):
    results = collection.query(
        query_texts=[query],
        n_results=10
    )

    docs = results["documents"][0]

    # Prioritize resume chunks so the model sees your actual skills
    resume_chunks = [
        d for d in docs
        if "MARCELA" in d or "TECHNICAL SKILLS" in d or "EDUCATION" in d or "SQL" in d
    ]
    jd_chunks = [d for d in docs if d not in resume_chunks]

    context = "\n\n".join(resume_chunks + jd_chunks)

    prompt = f"""
You are a job fit analyzer. Use ONLY the provided job description and resume context.

Task:
Create a Keyword Alignment analysis comparing the job description keywords against the resume.

Instructions:
- Extract 12 to 15 important keywords or phrases from the job description.
- For each keyword, determine whether it appears directly in the resume, appears as a semantic equivalent, or is missing.
- Do not mark SQL, Python, Tableau, Excel, analytics, or data visualization as missing if they appear anywhere in the resume context.
- Calculate an overall keyword match percentage:
  match percentage = (direct matches + semantic matches) / total keywords * 100
- Be specific and evidence-based.

Return the output in this structure:

1. Keyword Alignment Table
Columns:
- JD Keyword/Phrase
- Resume Match Status: Direct Match, Semantic Match, or Missing
- Resume Evidence
- Notes

2. Overall Keyword Match Percentage
Show the calculation.

3. Brief Interpretation
Write 3 to 4 sentences explaining what the match percentage means for the candidate's fit.

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You compare job description keywords to resume evidence using only provided text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [28]:
keyword_analysis = get_keyword_alignment(
    "Data Analyst job requirements keywords SQL Python Tableau Excel analytics pricing forecasting"
)

print(keyword_analysis)

### 1. Keyword Alignment Table

| JD Keyword/Phrase                                       | Resume Match Status | Resume Evidence                                                                 | Notes                                                                 |
|--------------------------------------------------------|---------------------|--------------------------------------------------------------------------------|-----------------------------------------------------------------------|
| Analytical skills                                       | Direct Match        | Strong analytical and problem-solving skills.                                 | Directly mentioned in the resume.                                    |
| Data analysis                                           | Direct Match        | Proficiency in Microsoft Excel and basic data analysis techniques.            | Directly mentioned in the resume.                                    |
| Communication skills          

In [29]:
# ── Analysis 3: Fit Summary ──
def get_fit_summary(query):
    results = collection.query(
        query_texts=[query],
        n_results=8
    )

    docs = results["documents"][0]

    # prioritize resume content
    resume_chunks = [
        d for d in docs
        if "MARCELA" in d or "TECHNICAL SKILLS" in d or "EDUCATION" in d or "SQL" in d
    ]
    jd_chunks = [d for d in docs if d not in resume_chunks]

    context = "\n\n".join(resume_chunks + jd_chunks)

    prompt = f"""
You are a hiring analyst evaluating candidate fit.

Task:
Provide a clear, concise Fit Summary comparing the candidate resume to the job description.

Instructions:
- Use ONLY the provided context.
- Be decisive and do not say "uncertain" or "assuming."
- Recognize existing strengths such as SQL, Python, Tableau, Excel, analytics experience if present.
- Focus on overall fit, not just listing skills.

Return the output in this structure:

1. Overall Fit Score (0–100)
- Give a numeric score
- Brief justification

2. Strengths
- 3–5 key strengths that align with the role

3. Weaknesses / Gaps
- 2–4 meaningful gaps (focus on real gaps, not tools already present)

4. Final Hiring Recommendation
- Strong Yes, Yes, Maybe, or No
- 2–3 sentence explanation

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You evaluate candidate fit using only provided resume and job description text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [30]:
fit_summary = get_fit_summary(
    "Data Analyst job fit SQL Python Tableau Excel analytics pricing forecasting"
)

print(fit_summary)

1. Overall Fit Score: 85
- The candidate demonstrates strong analytical skills and relevant technical expertise in SQL, Python, and Tableau, which align well with the job requirements. Their experience in analytics and data visualization positions them as a strong contender for the role, although they lack direct experience in the entertainment or gaming industry.

2. Strengths
- Proficient in SQL, Python, and Tableau, which are essential for data analysis and visualization.
- Extensive experience in building ETL pipelines and performing advanced analytics, including NLP and customer segmentation.
- Strong academic background with a high GPA and relevant certifications in Data Science and Business Analytics.
- Proven ability to translate complex data insights into actionable business recommendations, demonstrated through professional and project experiences.

3. Weaknesses / Gaps
- Lacks specific experience in the entertainment or gaming industry, which may limit understanding of indus

### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:**

Skill Gap Report:

The final prompt improved accuracy by forcing the model to check resume evidence before labeling a skill as missing. This corrected earlier false gaps for SQL and Tableau and produced more realistic gaps focused on advanced Excel, forecasting, pricing/revenue optimization, and business partnering.

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [31]:
# ── Few-shot version of your chosen analysis ──


In [32]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [33]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [34]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*